## For this problem, we will consider a manager an employee who has at least 1 other employee reporting to them.

Write a solution to report the ids and the names of all managers, the number of employees who report directly to them, and the average age of the reports rounded to the nearest integer.

Return the result table ordered by employee_id.

The result format is in the following example.

 

Example 1:

Input: 
Employees table:
+-------------+---------+------------+-----+
| employee_id | name    | reports_to | age |
+-------------+---------+------------+-----+
| 9           | Hercy   | null       | 43  |
| 6           | Alice   | 9          | 41  |
| 4           | Bob     | 9          | 36  |
| 2           | Winston | null       | 37  |
+-------------+---------+------------+-----+
Output: 
+-------------+-------+---------------+-------------+
| employee_id | name  | reports_count | average_age |
+-------------+-------+---------------+-------------+
| 9           | Hercy | 2             | 39          |
+-------------+-------+---------------+-------------+
Explanation: Hercy has 2 people report directly to him, Alice and Bob. Their average age is (41+36)/2 = 38.5, which is 39 after rounding it to the nearest integer.
Example 2:

Input: 
Employees table:
+-------------+---------+------------+-----+ 
| employee_id | name    | reports_to | age |
|-------------|---------|------------|-----|
| 1           | Michael | null       | 45  |
| 2           | Alice   | 1          | 38  |
| 3           | Bob     | 1          | 42  |
| 4           | Charlie | 2          | 34  |
| 5           | David   | 2          | 40  |
| 6           | Eve     | 3          | 37  |
| 7           | Frank   | null       | 50  |
| 8           | Grace   | null       | 48  |
+-------------+---------+------------+-----+ 
Output: 
+-------------+---------+---------------+-------------+
| employee_id | name    | reports_count | average_age |
| ----------- | ------- | ------------- | ----------- |
| 1           | Michael | 2             | 40          |
| 2           | Alice   | 2             | 37          |
| 3           | Bob     | 1             | 37          |
+-------------+---------+---------------+-------------+

## taking pandas data frame ftom leet code and convert them to Spark data Frame

In [1]:
import pandas as pd
data = [[9, 'Hercy', None, 43], [6, 'Alice', 9, 41], [4, 'Bob', 9, 36], [2, 'Winston', None, 37]]
employees = pd.DataFrame(data, columns=['employee_id', 'name', 'reports_to', 'age']).astype({'employee_id':'Int64', 'name':'object', 'reports_to':'Int64', 'age':'Int64'})

In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("employee report").getOrCreate()
emp_df=spark.createDataFrame(employees)
emp_df.show()

26/07/14 06:42:19 WARN Utils: Your hostname, os.local resolves to a loopback address: 127.0.0.1; using 192.168.31.208 instead (on interface en0)
26/07/14 06:42:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 06:42:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+-------+----------+---+
|employee_id|   name|reports_to|age|
+-----------+-------+----------+---+
|          9|  Hercy|       NaN| 43|
|          6|  Alice|       9.0| 41|
|          4|    Bob|       9.0| 36|
|          2|Winston|       NaN| 37|
+-----------+-------+----------+---+



In [ ]:
from pyspark.sql.functions import col,sum,count
emp=emp_df.alias("emp")
mng=emp_df.alias("mng")
df=(emp.join(mng,col("emp.reports_to")==col("mng.employee_id"), "inner").
withColumn("manager_name",col("mng.name"))
.withColumn("emp_age",col("emp.age"))
.withColumn("report_to", col("emp.reports_to").cast("int"))
).select("manager_name","emp_age","report_to")
df1=df.groupBy("report_to").avg(sum("emp_age")/count("*")).alias("avg_age")
df1.show()

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

26/07/15 07:10:03 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 676076 ms exceeds timeout 120000 ms
26/07/15 07:10:03 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/15 07:10:08 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [15]:
from pyspark.sql.functions import sum,col,count
df1=df.groupBy("emp.reports_to").agg(sum(col("age"))/count("*")).alias("avg_age")
df1.show()

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `age` is ambiguous, could be: [`emp`.`age`, `mng`.`age`].